In [1]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [6]:
!pip install python-dotenv

In [ ]:
!pip install langchain_community langchain chromadb pypdf tiktoken

In [4]:
import os
import json

from dotenv import load_dotenv
from openai import OpenAI

In [5]:
load_dotenv()


False

In [7]:
anyscale_api_key = os.environ['ANYSCALE_API_KEY']

client = OpenAI(
    base_url="https://api.endpoints.anyscale.com/v1",
    api_key=anyscale_api_key
)

KeyError: 'ANYSCALE_API_KEY'

In [10]:
# Load the JSON file and extract values
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    API_KEY = config.get("API_KEY") # Loading the API Key
    OPENAI_API_BASE = config.get("OPENAI_API_BASE") # Loading the API Base Url

model_name = "gpt-4o-mini"

# Storing API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

In [11]:
# you may use these too for trial.
llm = "meta-llama/Meta-Llama-3-8B-Instruct"
rater_model = "meta-llama/Meta-Llama-3-70B-Instruct"

In [12]:
context = """
### Context:
In 2020, we recognized total revenues of $31.54 billion, representing an increase of $6.96 billion compared to the prior year. We continue to ramp production, build new manufacturing capacity and expand our operations to enable increased deliveries and deployments of our products and further revenue growth. In 2020, our net income attributable to common stockholders was $721 million, representing a favorable change of $1.58 billion compared to the prior year. In 2020, our operating margin was 6.3%, representing a favorable change of 6.6% compared to the prior year. We continue to focus on operational efficiencies, while we have seen an acceleration of non-cash stock-based compensation expense due to a rapid increase in our market capitalization and updates to our business outlook. We ended 2020 with $19.38 billion in cash and cash equivalents, representing an increase of $13.12 billion from the end of 2019. Our cash flows from operating activities during 2020 was $5.94 billion, compared to $2.41 billion during 2019, and capital expenditures amounted to $3.16 billion during 2020, compared to $1.33 billion during 2019. Sustained growth has allowed our business to generally fund itself, but we will continue a number of capital-intensive projects in upcoming periods. Management Opportunities, Challenges and Risks and 2021 Outlook Impact of COVID-19 Pandemic.2020 compared to 2019 Automotive sales revenue increased $6.23 billion, or 31%, in the year ended December 31, 2020 as compared to the year ended December 31, 2019, primarily due to an increase of 129,268 Model 3 and Model Y cash deliveries despite production limitations as a result of temporary suspension of production at the Fremont Factory and Gigafactory Nevada during the first half of 2020. We were able to increase deliveries year over year from production ramping at both Gigafactory Shanghai and the Fremont Factory. There was also an increase of $986 million from additional sales of regulatory credits to $1.58 billion in the year ended December 31, 2020. Additionally, due to pricing adjustments we made to our vehicle offerings during the year ended December 31, 2019, we estimated that there was a greater likelihood that customers would exercise their buyback options and adjusted our sales return reserve on vehicles previously sold under our buyback options program which resulted in a reduction of automotive sales revenue of $555 million. We made further pricing adjustments that resulted in a similar but smaller reduction of automotive sales revenue of $72 million during the year ended December 31, 2020. The smaller reduction in revenue from pricing adjustments resulted in a positive impact to automotive sales revenue of $483 million year over year. These factors increasing automotive sales revenue were partially offset by a decrease in the combined average selling price of Model 3 and Model Y. Despite the inclusion of higher.These increases were partially offset by a decrease in used vehicle revenue driven by a reduction in non-Tesla trade-ins. Energy Generation and Storage Segment Energy generation and storage revenue includes sales and leasing of solar energy generation and energy storage products, services related to such products and sales of solar energy systems incentives. 2020 compared to 2019 Energy generation and storage revenue increased by $463 million, or 30%, in the year ended December 31, 2020 as compared to the year ended December 31, 2019, primarily due to increases in deployments of Megapack, solar cash and loan jobs and Powerwall, partially offset by a decrease in deployments of Powerpack and reduced average selling prices on our solar cash and loan jobs as a result of our low cost solar strategy. Powerpack deployments have decreased following the introduction of our Megapack product, which we began deploying in late 2019. 41.the year ended December 31, 2020. Additionally, there was an increase to cost of automotive sales revenue from idle capacity charges of $213 million as a result of temporary suspension of production at the Fremont Factory and Gigafactory Nevada during the first half of 2020. These factors increasing cost of automotive sales revenue were partially offset by a decrease in average Model 3 costs per unit due to lower material, manufacturing, freight and duty costs from localized procurement and manufacturing in China and a higher sales mix of lower end trims, as well as a decrease of 8,669 Model S and Model X cash deliveries in the year ended December 31, 2020 compared to the prior year. 42.R&D expenses increased $148 million , or 11% , in the year ended December 31, 2020 as compared to the year ended December 31, 2019 . The increase was primarily due to a $62 million increase in expensed materials as we continue to expand our product roadmap , $61 million increase in stock-based compensation expense primarily related to the issuance of equity awards in fiscal year 2020 at higher grant date fair values due to our increased share price, $20 million increase in facilities, freight and depreciation expenses and a $20 million increase in employee and labor related expenses. R&D expenses as a percentage of revenue decreased from 5.5% to 4.7% in the year ended December 31, 2020 as compared to the year ended December 31, 2019. The decrease is primarily an increase in total revenues from expanding sales, partially offset by an increase in our R&D expenses as detailed above. Selling, General and Administrative Expense Year Ended December 31, 2020 vs. 2019 Change 2019 vs. 2018 Change (Dollars in millions) 2020 2019 2018 $ % $ % Selling, general and administrative $ 3,145 $ 2,646 $ 2,835 $ 499 19 % $ (189 ) -7 %"""

question = """
### Question:
Did revenue grow in 2020 compared to the previous year? What was the main reason for this growth?
"""

answer = """
### Answer:
Yes, revenue grew in 2020 compared to the previous year.
The main reason for this growth was an increase of $6.96 billion in total revenues, primarily due to an increase of 31% in automotive sales revenue, and a $986 million increase from additional sales of regulatory credits.
"""

In [14]:
# Faithfulness

## Step 1

statement_generator_system_message = """
Given a question, an answer, and sentences from the answer, break down each sentence in the answer into one or more fully understandable statements while also ensuring no pronouns are used in each statement.
Format the outputs as a JSON list of statements.
DO NOT output anything else before or after the JSON list of statements.
"""

statement_prompt = [
    {'role': 'system', 'content': statement_generator_system_message},
    {'role': 'user', 'content': question + answer}
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=statement_prompt,
        temperature=0
    )
    statements = response.choices[0].message.content
except Exception as e:
    print(e)

print(statements)

[
    "Revenue grew in 2020 compared to the previous year.",
    "The main reason for this growth was an increase of $6.96 billion in total revenues.",
    "The increase in total revenues was primarily due to an increase of 31% in automotive sales revenue.",
    "There was a $986 million increase from additional sales of regulatory credits."
]


In [17]:
## Step 2

faithfullness_system_message = """
Your task is to judge the faithfulness of a series of statements based on a given context.
For each statement you must return verdict as 1 if the statement can be directly inferred based on the context or 0 if the statement can not be directly inferred based on the context.

Output Format:
Arrange your output in the following JSON format.
{
    "statement": statement from the list
    "explanation": <A step-by-step evaluation for faithfullness>
    "rating": integer either 1 or 0
}
DO NOT output anything else before or after the JSON output.
"""

faithfullness_prompt = [
    {'role': 'system', 'content': faithfullness_system_message},
    {'role': 'user', 'content': context + question + statements}
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=faithfullness_prompt,
        response_format={"type": "json_object"},
        temperature=0
    )
    rating = response.choices[0].message.content
except Exception as e:
    print(e)

print(rating)

{
    "statement": "Revenue grew in 2020 compared to the previous year.",
    "explanation": "The context states that total revenues in 2020 were $31.54 billion, which is an increase of $6.96 billion compared to the prior year. This confirms that revenue did grow in 2020 compared to 2019.",
    "rating": 1
}
  
  
  


In [23]:
# Context Precision

context_precision_system_message = """
Given question, answer and context verify if the context was useful in arriving at the given answer.
Give verdict as "1" if useful and "0" if not with a JSON output.

Output Format:
Arrange your output in the following JSON format.
{
    "explanation": <A step-by-step evaluation>
    "rating": integer either 1 or 0
}
DO NOT output anything else before or after the JSON output.
"""

context_precision_prompt = [
    {'role': 'system', 'content': context_precision_system_message},
    {'role': 'user', 'content': context + question + answer}
]

try:
    response = client.chat.completions.create(
        model=rater_model,
        messages=context_precision_prompt,
        response_format={"type": "json_object"},
        temperature=0
    )
    rating = response.choices[0].message.content
except Exception as e:
    print(e)

print(rating)

{
    "explanation": "The context provides specific figures indicating that total revenues increased by $6.96 billion in 2020 compared to the previous year. It also details that automotive sales revenue increased by 31%, contributing significantly to the overall revenue growth. Additionally, it mentions a $986 million increase from sales of regulatory credits, which further supports the answer. Therefore, the context was useful in confirming both the growth in revenue and the reasons behind it.",
    "rating": 1
}


The error came in the above line because we used rater_model which is not functioning in the notebook.

Try changing it to model = model_name

In [28]:
# Context Recall

context_recall_system_message = """
Given a context, and an answer, analyze each sentence in the answer and classify if the sentence can be attributed to the given context or not.
Use only "Yes" (1) or "No" (0) as a binary classification. Output json with reason.

Output Format:
Arrange your output in the following JSON format.
{
    "explanation": <A step-by-step evaluation>
    "rating": integer either 1 or 0
}
DO NOT output anything else before or after the JSON output.
"""

context_recall_prompt = [
    {'role': 'system', 'content': context_recall_system_message},
    {'role': 'user', 'content': context + question + answer}
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=context_recall_prompt,
        response_format={"type": "json_object"},
        temperature=0
    )
    rating = response.choices[0].message.content
except Exception as e:
    print(e)

print(rating)

{
    "explanation": "The first sentence confirms that revenue grew in 2020 compared to the previous year, which aligns with the context stating total revenues increased by $6.96 billion. The second sentence attributes the growth to an increase in automotive sales revenue and additional sales of regulatory credits, which is supported by the context that mentions a 31% increase in automotive sales revenue and a $986 million increase from regulatory credits.",
    "rating": 1
}


In [29]:
# Answer Relevance

answer_relevance_system_message = """
Generate a question for the given answer and identify if the answer is noncommittal.
Give noncommittal as 1 if the answer is noncommittal and 0 if the answer is committal.
A noncommittal answer is one that is evasive, vague, or ambiguous.
For example, "I don't know" or "I'm not sure" are noncommittal answers.

Output Format:
Arrange your output in the following JSON format.
{
    "explanation": <A step-by-step evaluation>
    "non_committal_rating": integer either 1 or 0
}
DO NOT output anything else before or after the JSON output.
"""

answer_relevance_prompt = [
    {'role': 'system', 'content': answer_relevance_system_message},
    {'role': 'user', 'content': context + question + answer}
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=answer_relevance_prompt,
        response_format={"type": "json_object"},
        temperature=0
    )
    rating = response.choices[0].message.content
except Exception as e:
    print(e)

print(rating)

{
    "explanation": "The answer clearly states that revenue grew in 2020 compared to the previous year and provides specific figures and reasons for this growth, indicating a definitive response to the question asked. There is no vagueness or ambiguity in the answer.",
    "non_committal_rating": 0
}
